# Exp 17 — Подготовка русских датасетов

Запускает `prepare.prepare_*` для каждого датасета и показывает результат.
Каждая функция парсит JSON-колонки в поля, делает multi-hot top-30 для массивов опций,
дропает `id` и константные колонки, сохраняет `data/raw_ru/<name>/clean.parquet`.

Если kagglehub просит логин — запусти `kagglehub.login()` в отдельной ячейке.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# чтобы `import prepare` сработал из ноутбука в experiments/17/
HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 200)

import prepare
from importlib import reload
reload(prepare)

<module 'prepare' from '/home/user/Lysenko/Diplom/TableUnifier/experiments/17/prepare.py'>

In [ ]:
def summarize(df: pd.DataFrame, name: str) -> None:
    print(f"=== {name} ===")
    print(f"shape: {df.shape}")
    print(f"memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

    def safe_nunique(s):
        try:
            return s.nunique(dropna=True)
        except TypeError:
            return -1

    stats = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.apply(safe_nunique),
        "n_null": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
    })
    print(stats.to_string())


def show_samples(df: pd.DataFrame, name: str, n: int = 1, max_len: int = 160) -> None:
    print(f"=== samples: {name} ===")
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]):
            continue
        vals = df[col].dropna().head(n).tolist()
        if not vals:
            continue
        print(f"\n· {col}")
        for v in vals:
            s = str(v).replace("\n", " ")
            if len(s) > max_len:
                s = s[:max_len] + "…"
            print(f"    {s}")

## 1. Lamoda

In [3]:
df_lamoda = prepare.prepare_lamoda()
summarize(df_lamoda, "lamoda")

=== lamoda ===
shape: (10338, 50)
memory: 24.6 MB
                                      dtype  n_unique  n_null  null_%
Title                                object      1979       0     0.0
Brand                                object      1093       0     0.0
Price                                object      2707    3373    32.6
Image                                object     10337       0     0.0
about.Состав, %                      object      2407      36     0.3
about.Сезон                          object         9      53     0.5
about.Размер товара на модели        object       303    2077    20.1
about.Длина по внутреннему шву (см)  object        69    8742    84.6
about.Длина по боковому шву          object        88    8807    85.2
about.Ширина по низу                 object        40    9332    90.3
about.Высота                         object        23    9635    93.2
about.Параметры модели               object       587    5006    48.4
about.Артикул                        obj

In [4]:
show_samples(df_lamoda, "lamoda")

=== samples: lamoda ===

· Title
    Брюки спортивные

· Brand
    O'STIN

· Price
    680 ₽

· Image
    //a.lmcdn.ru/img600x866/M/P/MP002XM0BBEG_24488927_1_v1_2x.jpg

· about.Состав, %
    Хлопок - 60%, Полиэстер - 40%

· about.Сезон
    мульти

· about.Размер товара на модели
    M INT

· about.Длина по внутреннему шву (см)
    71 см

· about.Длина по боковому шву
    103 см

· about.Ширина по низу
    17 см

· about.Высота
    35 см

· about.Параметры модели
    100 - 84 -94

· about.Артикул
    MP002XM0BBEG

· about.Рост модели на фото
    189 см

· about.Материал подкладки, %
    без подкладки

· about.Цвет
    черный

· about.Узор
    однотонный

· about.Застежка
    пуговицы

· about.Страна производства
    Китай

· about.Гарантийный срок
    30 дней

· about.Длина (см)
    65 см

· about.Длина рукава (см)
    65 см

· about.Вырез/воротник
    поло

· about.Комплектация
    Футболка

· about.Экосостав
    данный товар имеет натуральный или органический состав

· about.Флисовая 

In [5]:
df_lamoda.head(2)

,Title,Brand,Price,Image,"about.Состав, %",about.Сезон,about.Размер товара на модели,about.Длина по внутреннему шву (см),about.Длина по боковому шву,about.Ширина по низу,about.Высота,about.Параметры модели,about.Артикул,about.Рост модели на фото,"about.Материал подкладки, %",about.Цвет,about.Узор,about.Застежка,about.Страна производства,about.Гарантийный срок,about.Длина (см),about.Длина рукава (см),about.Вырез/воротник,about.Комплектация,about.Экосостав,about.Флисовая подкладка,about.Утеплитель,about.Фасон,about.Вид спорта,about.Сезон 2,about.Тип трикотажа,about.Рисунок,about.Тип силуэта,about.Внешние карманы,about.Тип ткани,about.Детали,about.Бережно к животным,about.Внутренние карманы,about.Капюшон,about.Технологии,about.Мембрана,about.Посадка,about.Тип плавок,about.Фасон брюк,about.Косточки,about.Чашечки,about.Бретели,about.Push up,about.Количество Den,about.Тип юбки
0,Брюки спортивные,O'STIN,680 ₽,//a.lmcdn.ru/img600x866/M/P/MP002XM0BBEG_24488927_1_v1_2x.jpg,"Хлопок - 60%, Полиэстер - 40%",мульти,M INT,71 см,103 см,17 см,35 см,100 - 84 -94,MP002XM0BBEG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Брюки спортивные,O'STIN,680 ₽,//a.lmcdn.ru/img600x866/M/P/MP002XM0BBEL_24107566_1_v1_2x.jpg,"Хлопок - 60%, Полиэстер - 40%",мульти,M INT,71 см,100 см,NaN,31 см,103 - 78 - 98,MP002XM0BBEL,189 см,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. cars_ru (market)

In [6]:
df_cars_ru = prepare.prepare_cars_ru()
summarize(df_cars_ru, "cars_ru")

=== cars_ru ===
shape: (100000, 54)
memory: 339.3 MB
                            dtype  n_unique  n_null  null_%
price_rub                 float64      3257   19991    20.0
mark                       object       193       0     0.0
model                      object      1890       0     0.0
generation                 object      1082    9862     9.9
configuration              object       268       0     0.0
complectation              object      2642   68456    68.5
body_type                  object        26       0     0.0
color                      object        16       8     0.0
displacement              float64        73       0     0.0
drive_type                 object         3       0     0.0
engine_type                object         5       0     0.0
horse_power                 int64       470       0     0.0
transmission               object         4       0     0.0
wheel                      object         2       0     0.0
km_age                    float64     16098    

In [7]:
show_samples(df_cars_ru, "cars_ru")

=== samples: cars_ru ===

· mark
    Kia

· model
    Sportage

· generation
    IV Рестайлинг

· configuration
    Внедорожник 5 дв.

· complectation
    Prestige Black Edition

· body_type
    пятидверный внедорожник

· color
    чёрный

· drive_type
    полный

· engine_type
    бензин

· transmission
    автомат

· wheel
    левый

· condition
    среднее

· pts
    оригинал

· seller_type
    частник

· region
    Вологодская область

· city
    Череповец

· address
    Россия, Вологодская область, 

· description
    Автомобиль в отличном состоянии, вложений никаких не требует. 2,4 атмосферный двигатель на обычном автомате Полный привод! Куплен не в кредит. Своевременно обсл…


In [8]:
df_cars_ru.head(2)

,price_rub,mark,model,generation,configuration,complectation,body_type,color,displacement,drive_type,engine_type,horse_power,transmission,wheel,km_age,year,owners_count,condition,pts,seller_type,region,city,address,description,opt_airbag-driver,opt_electro-window-front,opt_lock,opt_immo,opt_abs,opt_wheel-configuration1,opt_airbag-passenger,opt_electro-mirrors,opt_mirrors-heat,opt_seat-transformation,opt_computer,opt_electro-window-back,opt_front-seats-heat,opt_isofix,opt_wheel-power,opt_wheel-configuration2,opt_airbag-side,opt_esp,opt_ptf,opt_fabric-seats,opt_audiosystem-cd,opt_wheel-leather,opt_multi-wheel,opt_usb,opt_bluetooth,opt_airbag-curtain,opt_cruise-control,opt_front-centre-armrest,opt_12v-socket,opt_light-sensor
0,2200000.0,Kia,Sportage,IV Рестайлинг,Внедорожник 5 дв.,Prestige Black Edition,пятидверный внедорожник,чёрный,2.4,полный,бензин,184,автомат,левый,51150.0,2020,1,среднее,оригинал,частник,Вологодская область,Череповец,"Россия, Вологодская область,","Автомобиль в отличном состоянии, вложений никаких не требует. 2,4 атмосферный двигатель на обычном автомате Полный привод! Куплен не в кредит. Своевременно обслуживался. Прошёл все ТО. Есть сервис...",True,True,True,True,True,True,True,False,False,True,True,True,False,True,False,True,True,True,True,False,True,True,True,False,True,True,True,True,True,True
1,300000.0,Ford,Maverick,III,Внедорожник 5 дв.,NaN,пятидверный внедорожник,серебристый,3.0,полный,бензин,197,автомат,левый,250000.0,2001,4,среднее,дубликат,частник,Краснодарский край,Сочи,Теневой переулок,"Есть жизненные повреждения, автомобиль рабочий, состояние соответствующее, по ходовой и технике все хорошо, езжу каждый день. руки естественно приложить есть куда",False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## 3. Ozon favorites

In [9]:
df_ozon = prepare.prepare_ozon()
summarize(df_ozon, "ozon")

/home/user/Lysenko/Diplom/TableUnifier/experiments/17/prepare.py:178: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(parts, ignore_index=True)


=== ozon ===
shape: (333738, 14)
memory: 382.2 MB
                                                          dtype  n_unique  n_null  null_%
Название товара                                          object    170160     946     0.3
Бренд                                                    object     20991   27500     8.2
Ссылка на товар                                          object    198493       0     0.0
Категория 1 уровня                                       object        25       0     0.0
Категория 2 уровня                                       object       188       0     0.0
Категория 3 уровня                                       object       915       0     0.0
Категория 4 уровня                                       object      5003       0     0.0
Количество добавлений в избранное, 30 дней              float64      1792   23749     7.1
Количество добавлений в избранное, все время     datetime64[ns]       222  323738    97.0
Последнее появление в наличии                    d

In [10]:
show_samples(df_ozon, "ozon")

=== samples: ozon ===

· Название товара
    Дневник школьный, Кошечка, розовый

· Бренд
    bumbel

· Ссылка на товар
    https://www.ozon.ru/product/244927827

· Категория 1 уровня
    Товары для мам и детей

· Категория 2 уровня
    Товары для школы, канцелярия, оборудование

· Категория 3 уровня
    Бумага офисная и бумажная продукция

· Категория 4 уровня
    Школьный дневник

· Количество добавлений в избранное, все время
    2021-06-21 00:00:00

· Последнее появление в наличии
    2020-12-09 00:00:00

· __source_file
    2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx


In [11]:
df_ozon.head(2)

,Название товара,Бренд,Ссылка на товар,Категория 1 уровня,Категория 2 уровня,Категория 3 уровня,Категория 4 уровня,"Количество добавлений в избранное, 30 дней","Количество добавлений в избранное, все время",Последнее появление в наличии,__source_file,"Количество добавлений в избранное, декабрь 2020","Количество добавлений в избранное, ноябрь 2020","Количество добавлений в избранное, январь 2021"
0,"Дневник школьный, Кошечка, розовый",bumbel,https://www.ozon.ru/product/244927827,Товары для мам и детей,"Товары для школы, канцелярия, оборудование",Бумага офисная и бумажная продукция,Школьный дневник,3280.0,2021-06-21,NaT,2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx,NaN,NaN,NaN
1,"Игровая консоль PlayStation 5 Digital Edition, белый",PlayStation,https://www.ozon.ru/product/178715781,ТВ и аудио,Игровые приставки и аксессуары TV&Audio,Игровая приставка TV&Audio,Игровая приставка,3059.0,2021-06-24,NaT,2021-07-12_opendata_datasetfavorites_2021-06-12_2021-07-11.xlsx,NaN,NaN,NaN


## 4. auto.ru parsed

In [12]:
df_auto_ru = prepare.prepare_auto_ru()
summarize(df_auto_ru, "auto_ru")

/home/user/Lysenko/Diplom/TableUnifier/experiments/17/prepare.py:61: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


TypeError: unhashable type: 'list'

In [13]:
show_samples(df_auto_ru, "auto_ru")

NameError: name 'df_auto_ru' is not defined

In [14]:
df_auto_ru.head(2)

NameError: name 'df_auto_ru' is not defined

## 5. auto.ru 14-11-2020

In [15]:
df_auto_ru_2020 = prepare.prepare_auto_ru_2020()
summarize(df_auto_ru_2020, "auto_ru_2020")

TypeError: unhashable type: 'list'

In [16]:
show_samples(df_auto_ru_2020, "auto_ru_2020")

NameError: name 'df_auto_ru_2020' is not defined

In [17]:
df_auto_ru_2020.head(2)

NameError: name 'df_auto_ru_2020' is not defined

## 6. DeviceStatus 15K

In [18]:
df_devices = prepare.prepare_devices()
summarize(df_devices, "devices")

=== devices ===
shape: (15000, 17)
memory: 9.5 MB
                              dtype  n_unique  n_null  null_%
device_model                 object      9611       0     0.0
model_family                 object        18       0     0.0
manufacturer                 object        10       0     0.0
device_type                  object        19       0     0.0
release_year                  int64        19       0     0.0
status                       object         2       0     0.0
usage_intensity              object         3       0     0.0
climate_zone                 object         3       0     0.0
technical_specs.resolution   object         3   12682    84.5
start_year                  float64        19    7443    49.6
service_life_years          float64         6    7443    49.6
predicted_break_year        float64        24    7443    49.6
actual_break_year           float64        26    7443    49.6
technical_specs.ports       float64         4   11864    79.1
technical_specs.conn

In [19]:
show_samples(df_devices, "devices")

=== samples: devices ===

· device_model
    ПРО-352

· model_family
    ПРО-Series

· manufacturer
    Lenovo

· device_type
    Projector

· status
    in stock

· usage_intensity
    low

· climate_zone
    moderate

· technical_specs.resolution
    Full HD

· technical_specs.connection
    Bluetooth

· technical_specs.type
    Condenser


In [20]:
df_devices.head(2)

,device_model,model_family,manufacturer,device_type,release_year,status,usage_intensity,climate_zone,technical_specs.resolution,start_year,service_life_years,predicted_break_year,actual_break_year,technical_specs.ports,technical_specs.connection,technical_specs.type,technical_specs.dpi
0,ПРО-352,ПРО-Series,Lenovo,Projector,2013,in stock,low,moderate,Full HD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ВЕБ-192,ВЕБ-Series,Logitech,Webcam,2016,in stock,medium,hot,1080p,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Итог

Все 6 датасетов лежат в `data/raw_ru/<name>/clean.parquet`.
Дальше — блокинг кандидатов + разметка (exp 13-стайл LLM-агентом).